# Define & Profile a Stabilizer Code

<div style="border-left:4px solid #6d3f8c;background:#6d3f8c1a;border-radius:4px;margin:1em 0;"><div style="background:#6d3f8c;color:#ffffff;padding:0.35em 0.8em;font-weight:600;"><span style="display:inline-block;width:1.15em;height:1.15em;line-height:1.15em;border-radius:50%;background:#ffffff;color:#6d3f8c;text-align:center;font-weight:700;margin-right:0.5em;">!</span>Chapter focus</div>

<div style="padding:0.1em 1em;">

How do you specify a stabilizer code in `qodec.Code` and verify its $[[n, k, d]]$, pure-$X$/$Z$ distances, and syndromes with `CodeProfile`?

</div>

</div>

## Learning objectives

After completing this unit, you will be able to:

- Construct a `qodec.Code` from indexed sparse Pauli strings and logical operator lists.
- Use `ec.CodeProfile` to compute overall distance, biased ($X$-only / $Z$-only) distances, and minimum-weight logical operators.
- Query error syndromes (`syndrome_of`) and logical effects (`logical_effect_of`) on candidate stabilizer codes.

## Before you begin

This course requires `qdk[ec]>=1.33.0`, `qodec>=0.2.0`, `paulimer`, `binar`, and `ipykernel`. Run the cell below to verify your active Python kernel and environment.

In [ ]:
# If packages are missing, first select a dedicated Python environment/kernel,
# then uncomment the next line and run this cell again.
# %pip install -r ../requirements.txt

from _unit import check_env

check_env()

In [2]:
import qodec
import qdk.ec as ec
from _unit import simulation

# Define the [[4, 2, 2]] C4 Code & Inspect with CodeProfile
c4 = qodec.Code(
    "C4",
    stabilizers=["X_0 X_1 X_2 X_3", "Z_0 Z_1 Z_2 Z_3"],
    x=["X_0 X_1", "X_0 X_2"],
    z=["Z_0 Z_2", "Z_0 Z_1"],
)

c4_profile = ec.CodeProfile(c4)
dist_all = c4_profile.distance()
dist_z_only = c4_profile.distance(errors="Z")
dist_obs0 = c4_profile.distance(logical_observable=ec.Pauli("X_0 X_1"))

print(f"Code name          : {c4.name}")
print(f"Stabilizers        : {[str(s) for s in c4.stabilizers]}")
print(f"Logical X ops      : {[str(x) for x in c4.x]}")
print(f"Logical Z ops      : {[str(z) for z in c4.z]}")
print(f"\nUnrestricted distance : d = {dist_all.value} (is_exact={dist_all.is_exact})")
print(f"  Witness product     : {dist_all.witness.product}")
print(f"  Witness factors     : {dist_all.witness.factors}")
print(f"Z-noise only distance : d_Z = {dist_z_only.value} (witness: {dist_z_only.witness.product})")
print(f"Distance for X_0 X_1  : d   = {dist_obs0.value} (witness: {dist_obs0.witness.product})")

Code name          : C4
Stabilizers        : ['X_0 X_1 X_2 X_3', 'Z_0 Z_1 Z_2 Z_3']
Logical X ops      : ['X_0 X_1', 'X_0 X_2']
Logical Z ops      : ['Z_0 Z_2', 'Z_0 Z_1']

Unrestricted distance : d = 2 (is_exact=True)
  Witness product     : XX
  Witness factors     : (X, IX)
Z-noise only distance : d_Z = 2 (witness: ZZ)
Distance for X_0 X_1  : d   = 2 (witness: YY)


## Querying Syndromes and Logical Effects with `CodeProfile`

Given any physical Pauli error $E$, `ec.CodeProfile` lets us immediately ask three fundamental questions:
1. **`c4_profile.syndrome_of(E)`**: Returns the set of **0-based stabilizer generator indices** in `code.stabilizers` that anti-commute with $E$ (for $C_4$: index `0` is $g_X = X_0 X_1 X_2 X_3$, and index `1` is $g_Z = Z_0 Z_1 Z_2 Z_3$).
2. **`c4_profile.logical_effect_of(E)`**: Returns which **logical Pauli generators** ($X_0, X_1, Z_0, Z_1$) **anti-commute** with $E$.
   - *QEC Researcher Note:* Why does `logical_effect_of(ec.Pauli("X_0 X_1"))` return `['Z_0']` rather than `['X_0']`? Because `X_0 X_1` *is* the logical operator $\bar{X}_0$, which **anti-commutes with and flips the measurement outcome of logical observable $\bar{Z}_0$**!
3. **`c4_profile.is_logical(E)`**: Returns `True` if $E$ commutes with all stabilizers (`syndrome_of(E)` is empty) while acting non-trivially on the logical space.

In [3]:
# Querying Syndromes, Logical Effects, and Undetectable Logical Errors
from _unit import simulation
simulation.ensure_course_state(globals(), stage="code")

stab_labels = {idx: str(s) for idx, s in enumerate(c4.stabilizers)}
print("Stabilizer generator index map:", stab_labels)

test_errors = ["X_1", "Z_2", "Y_0", "X_0 X_1", "X_2 X_3", "X_0 X_1 X_2 X_3"]

print(f"\n{'Pauli Error':<18} | {'Violated Check Indices':<24} | {'Flipped Logical Obs':<24} | {'is_logical?'}")
print("-" * 86)
for err in test_errors:
    p_err = ec.Pauli(err)
    syn = sorted(c4_profile.syndrome_of(p_err))
    is_log = c4_profile.is_logical(p_err)
    eff = str(c4_profile.logical_effect_of(p_err)) if len(syn) == 0 else "(detected by syndrome)"
    print(f"{err:<18} | {str(syn):<24} | {eff:<24} | {is_log}")

Stabilizer generator index map: {0: 'X_0 X_1 X_2 X_3', 1: 'Z_0 Z_1 Z_2 Z_3'}

Pauli Error        | Violated Check Indices   | Flipped Logical Obs      | is_logical?
--------------------------------------------------------------------------------------
X_1                | [1]                      | (detected by syndrome)   | False
Z_2                | [0]                      | (detected by syndrome)   | False
Y_0                | [0, 1]                   | (detected by syndrome)   | False
X_0 X_1            | []                       | X                        | True
X_2 X_3            | []                       | X                        | True
X_0 X_1 X_2 X_3    | []                       | I                        | False


Notice two critical insights from the table above:
- **Any weight-1 Pauli error** (`X_1`, `Z_2`, `Y_0`) produces a non-zero syndrome (`is_logical = False`), confirming that $C_4$ detects all single-qubit errors ($d = 2$).
- **Weight-2 operator `X_2 X_3`** has `Violated Check Indices = []` and `Flipped Logical Obs = ['Z_0']` (`is_logical = True`), because $X_2 X_3 = (X_0 X_1) \cdot (X_0 X_1 X_2 X_3)$ differs from $\bar{X}_0 = X_0 X_1$ by the $X$-stabilizer!

## Build and Profile the $[[5, 1, 3]]$ Perfect Code

In this hands-on exercise, you will construct the **$[[5, 1, 3]]$ Laflamme 5-qubit code** (`FiveQubit_5_1_3`, the smallest stabilizer code that corrects an arbitrary single-qubit error) and use `ec.CodeProfile` to compare its distance under two different physical noise models:
1. **Step A:** Define `five_qubit_code = qodec.Code("FiveQubit_5_1_3", ...)` on qubits `0..4` with the 4 cyclic $X Z Z X I$ stabilizer generators:
   - `"X_0 Z_1 Z_2 X_3"`, `"X_1 Z_2 Z_3 X_4"`, `"X_0 X_2 Z_3 Z_4"`, `"Z_0 X_1 X_3 Z_4"`
   and weight-5 logical operators `x=["X_0 X_1 X_2 X_3 X_4"]`, `z=["Z_0 Z_1 Z_2 Z_3 Z_4"]`.
2. **Step B:** Create `five_profile = ec.CodeProfile(five_qubit_code)` and compute:
   - `d_depol = five_profile.distance(errors="XYZ")` (standard depolarizing noise, where $d = 3$), and
   - `d_z_bias = five_profile.distance(errors="Z")` (infinitely $Z$-biased dephasing noise, where $d_Z = 5$ because the only pure-$Z$ non-trivial logical operator is $Z_0 Z_1 Z_2 Z_3 Z_4$!).

> **💡 Using This Exercise with GitHub Copilot:** Try modifying the `errors` filter (e.g. `errors="X"` or `errors="XZ"`) or removing one stabilizer generator to see how `CodeProfile.distance()` responds. Expand the reference block below if you want to check the exact solution.

In [4]:
# Worked Example — Define the Non-CSS [[5, 1, 3]] Five-Qubit Code and Inspect d and d_X
five_qubit_code = qodec.Code(
    "C5",
    stabilizers=[
        "X_0 Z_1 Z_2 X_3",
        "X_1 Z_2 Z_3 X_4",
        "X_0 X_2 Z_3 Z_4",
        "Z_0 X_1 X_3 Z_4",
    ],
    x=["X_0 X_1 X_2 X_3 X_4"],
    z=["Z_0 Z_1 Z_2 Z_3 Z_4"],
)
five_profile = ec.CodeProfile(five_qubit_code)
d_all = five_profile.distance()
d_x_bias = five_profile.distance(errors="X")

print(f"[[5, 1, 3]] Code -> {len(five_qubit_code.stabilizers)} stabilizers, {len(five_qubit_code.z)} logical qubit(s)")
print(f"  Unbiased distance d       = {d_all.value}   (witness: {d_all.witness.product})")
print(f"  Pure-X biased distance dX = {d_x_bias.value}   (witness: {d_x_bias.witness.product})")

[[5, 1, 3]] Code -> 4 stabilizers, 1 logical qubit(s)
  Unbiased distance d       = 3   (witness: IIZXZ)
  Pure-X biased distance dX = 5   (witness: XXXXX)


## `[EXTENSION: Fire-and-Ice]` Combining "Fire" ($[[5, 1, 3]]$) and "Ice" ($[[4, 2, 2]]$) into the $[[20, 2, 6]]$ Fire-and-Ice Code

> **Modular Extension ([arXiv:2605.15344 ↗](https://arxiv.org/abs/2605.15344), Reichardt, Aasen & Chao — *"Fire and ice: Partially fault-tolerant quantum computing with selective state filtering"*):**
> Look back at the two codes you just defined in Module 1:
> 1. In Section 1.2, you defined the **4-qubit $[[4, 2, 2]]$ Iceberg code ("Ice")**.
> 2. In Section 1.4 (Lab 1), you defined the **5-qubit $[[5, 1, 3]]$ Laflamme code ("Fire")**.
>
> In [arXiv:2605.15344 ↗](https://arxiv.org/abs/2605.15344), Reichardt, Aasen, and Chao show how concatenating an outer ququad stabilizer code over **inner $[[4, 2, 2]]$ Iceberg blocks** produces a family of high-rate, self-dual CSS codes (Figure 4 of the paper):
>
> | Concatenated Code Family | Outer Code ("Top") | Inner Code ("Bottom") | Parameters $[[n, k, d]]$ | Literature Name |
> | :--- | :--- | :--- | :--- | :--- |
> | $[[3, 1, 2]]_4 \circ [[4, 2, 2]]$ | 3-ququad $[[6, 2, 2]]$ | 3 blocks of $[[4, 2, 2]]$ | **$[[12, 2, 4]]$** | Knill $C_4/C_6$ / **Carbon code** |
> | $[[4, 2, 2]] \circ [[4, 2, 2]]$ | $C_4$ product | 4 blocks of $[[4, 2, 2]]$ | **$[[16, 4, 4]]$** | $C_4 \times C_4$ / **Many-hypercube code** |
> | **$[[5, 1, 3]]_4 \circ [[4, 2, 2]]$** | **5-ququad lifted $[[5, 1, 3]]$ ("Fire")** | **5 blocks of $[[4, 2, 2]]$ ("Ice")** | **$[[20, 2, 6]]$** | **Fire-and-Ice code** ([arXiv:2605.15344 ↗](https://arxiv.org/abs/2605.15344), Fig. 2) |
>
> #### How Lemma 1 & Corollary 2 Construct the $[[20, 2, 6]]$ Fire-and-Ice Code
> How can the non-CSS 5-qubit code ($X Z Z X I$ cyclic shifts) become a **self-dual CSS code**?
> - **Lemma 1 (Doubling to $[[10, 2, 3]]$):** Split each 5-qubit stabilizer generator $g = (g_X \mid g_Z)$ into two 5-qubit halves $A_j$ and $B_j$: place the $X$-support on half $A$ and the $Z$-support on half $B$ (all as $X$-type checks, and identically as $Z$-type checks), then swap the two halves (`SWAP(A_j, B_j)`). Because $X Z Z X I$ has two $X$'s and two $Z$'s per generator, every doubled stabilizer has weight $2 + 2 = 4$.
> - **Corollary 2 (Concatenating onto 5 blocks of $[[4, 2, 2]]$):** Each of the $j \in \{0, 1, 2, 3, 4\}$ blocks is a 4-physical-qubit $[[4, 2, 2]]$ Iceberg code on qubits `{4j, 4j+1, 4j+2, 4j+3}`, where the two logical qubits $(A_j, B_j)$ of block $j$ have weight-2 logical operators:
>   - First logical qubit $A_j$: $\bar{X}_{j,0} = X_{4j} X_{4j+1}$ (and $\bar{Z}_{j,1} = Z_{4j} Z_{4j+1}$)
>   - Second logical qubit $B_j$: $\bar{X}_{j,1} = X_{4j} X_{4j+2}$ (and $\bar{Z}_{j,0} = Z_{4j} Z_{4j+2}$)
> - Substituting these weight-2 block operators into the 8 weight-4 outer stabilizers yields **8 weight-8 outer CSS stabilizers** plus **10 weight-4 inner $[[4, 2, 2]]$ stabilizers** on $n = 20$ physical qubits (`0..19`), encoding **$k = 2$ logical qubits with exact code distance $d = 2 \times 3 = 6$**!

Run the cell below to construct the **$[[20, 2, 6]]$ Fire-and-Ice code** programmatically in `qodec.Code` and verify its exact CSS distance $d_X = d_Z = 6$ using `ec.CodeProfile`.

In [5]:
# Fire-and-Ice Extension: Construct and Profile the [[20, 2, 6]] Fire-and-Ice Code (arXiv:2605.15344)
from _unit import simulation
simulation.ensure_course_state(globals(), stage="code")

# 5 inner [[4, 2, 2]] Iceberg ("Ice") blocks on qubits {4j, 4j+1, 4j+2, 4j+3} for j = 0..4:
inner_ice_stabs = []
for j in range(5):
    q0, q1, q2, q3 = 4 * j, 4 * j + 1, 4 * j + 2, 4 * j + 3
    inner_ice_stabs.append(f"X_{q0} X_{q1} X_{q2} X_{q3}")
    inner_ice_stabs.append(f"Z_{q0} Z_{q1} Z_{q2} Z_{q3}")

# Outer [[5, 1, 3]] Laflamme ("Fire") cyclic generators on 5 blocks:
# Generator s in {0..3} has X on blocks (s, s+3 mod 5) and Z on blocks (s+1, s+2 mod 5).
# By Lemma 1 + Corollary 2 (Fig. 2 of arXiv:2605.15344), lifting onto the two logical qubits
# (pair {4j, 4j+1} and pair {4j, 4j+2}) of each [[4, 2, 2]] block yields 8 weight-8 CSS stabilizers:
outer_fire_stabs = []
for p in ("X", "Z"):
    for s in range(4):
        x_blocks = [s % 5, (s + 3) % 5]
        z_blocks = [(s + 1) % 5, (s + 2) % 5]
        terms = [f"{p}_{4*j} {p}_{4*j+1}" for j in x_blocks] + [f"{p}_{4*j} {p}_{4*j+2}" for j in z_blocks]
        outer_fire_stabs.append(" ".join(terms))

# Logical operators for the k = 2 encoded qubits:
fni_logical_x = [
    " ".join(f"X_{4*j} X_{4*j+1}" for j in range(5)),
    " ".join(f"X_{4*j} X_{4*j+2}" for j in range(5)),
]
fni_logical_z = [
    " ".join(f"Z_{4*j} Z_{4*j+2}" for j in range(5)),
    " ".join(f"Z_{4*j} Z_{4*j+1}" for j in range(5)),
]

fire_and_ice = qodec.Code(
    "FireAndIce_20_2_6",
    stabilizers=inner_ice_stabs + outer_fire_stabs,
    x=fni_logical_x,
    z=fni_logical_z,
)

fni_profile = ec.CodeProfile(fire_and_ice)
d_fni_z = fni_profile.distance(errors="Z")
d_fni_x = fni_profile.distance(errors="X")

print(f"Code               : {fire_and_ice.name} ([[20, 2, 6]] self-dual CSS code)")
print(f"Stabilizer count   : {len(fire_and_ice.stabilizers)} (10 weight-4 Ice checks + 8 weight-8 Fire checks)")
print(f"Logical qubits (k) : {len(fire_and_ice.x)}")
print(f"Exact CSS distance : d_Z = {d_fni_z.value} (witness: {d_fni_z.witness.product})")
print(f"                     d_X = {d_fni_x.value} (witness: {d_fni_x.witness.product})")
assert d_fni_z.value == 6 and d_fni_x.value == 6, "Expected exact distance 6 for [[20, 2, 6]] Fire-and-Ice code!"

Code               : FireAndIce_20_2_6 ([[20, 2, 6]] self-dual CSS code)
Stabilizer count   : 18 (10 weight-4 Ice checks + 8 weight-8 Fire checks)
Logical qubits (k) : 2
Exact CSS distance : d_Z = 6 (witness: IIIIIZIZIIIIIIZZIIZZ)
                     d_X = 6 (witness: IIXXIXIXIIIIIIIIIXIX)


## Knowledge check

<div style="border-left:4px solid #8c4a00;background:#8c4a001a;border-radius:4px;margin:1em 0;"><details><summary style="background:#8c4a00;color:#ffffff;padding:0.35em 0.8em;cursor:pointer;font-weight:600;">&#10067;&nbsp;In <code style="color:inherit;font-family:var(--vscode-editor-font-family,monospace);background:rgba(0,0,0,0.2);border-radius:3px;padding:0 0.2em;">c4 = qodec.Code('C4', ...)</code>, why does the physical error <code style="color:inherit;font-family:var(--vscode-editor-font-family,monospace);background:rgba(0,0,0,0.2);border-radius:3px;padding:0 0.2em;">X_2 X_3</code> have <code style="color:inherit;font-family:var(--vscode-editor-font-family,monospace);background:rgba(0,0,0,0.2);border-radius:3px;padding:0 0.2em;">c4_profile.is_logical('X_2 X_3') == True</code> even though <code style="color:inherit;font-family:var(--vscode-editor-font-family,monospace);background:rgba(0,0,0,0.2);border-radius:3px;padding:0 0.2em;">X_2 X_3</code> is not listed directly in <code style="color:inherit;font-family:var(--vscode-editor-font-family,monospace);background:rgba(0,0,0,0.2);border-radius:3px;padding:0 0.2em;">c4.x</code>?</summary>

<div style="padding:0.1em 1em;">

Because `X_2 X_3` equals the product of the first logical X generator (`X_0 X_1`) and the X stabilizer (`X_0 X_1 X_2 X_3`). Stabilizer codes define logical operators up to multiplication by any element of the stabilizer group.

</div>

</details></div>

<div style="border-left:4px solid #8c4a00;background:#8c4a001a;border-radius:4px;margin:1em 0;"><details><summary style="background:#8c4a00;color:#ffffff;padding:0.35em 0.8em;cursor:pointer;font-weight:600;">&#10067;&nbsp;How do you query the minimum-weight logical operator of a code when physical noise is purely dephasing (<i>Z</i>-only)?</summary>

<div style="padding:0.1em 1em;">

Call `ec.CodeProfile(code).distance(errors='Z')`, which restricts the fault-generating set to $Z$ Pauli operators on each qubit.

</div>

</details></div>

## Exercise: Pure-Z Distance of the [[5, 1, 3]] Five-Qubit Code

Construct a `CodeProfile` for the $[[5, 1, 3]]$ five-qubit code and return its pure-$Z$ code distance (`errors='Z'`).

In [ ]:
from qdk import ec
import qodec
from _unit import exercise


@exercise
def five_qubit_z_distance() -> int:
    five_qubit_code = qodec.Code(
        'C5',
        stabilizers=[
            'X_0 Z_1 Z_2 X_3',
            'X_1 Z_2 Z_3 X_4',
            'X_0 X_2 Z_3 Z_4',
            'Z_0 X_1 X_3 Z_4',
        ],
        x=['X_0 X_1 X_2 X_3 X_4'],
        z=['Z_0 Z_1 Z_2 Z_3 Z_4'],
    )
    # Return the pure-Z code distance of five_qubit_code
    return ...

**Hint**

Call `ec.CodeProfile(five_qubit_code).distance(errors='Z').value`.

In [ ]:
from qdk import ec
import qodec
from _unit import exercise


@exercise
def five_qubit_z_distance() -> int:
    five_qubit_code = qodec.Code(
        'C5',
        stabilizers=[
            'X_0 Z_1 Z_2 X_3',
            'X_1 Z_2 Z_3 X_4',
            'X_0 X_2 Z_3 Z_4',
            'Z_0 X_1 X_3 Z_4',
        ],
        x=['X_0 X_1 X_2 X_3 X_4'],
        z=['Z_0 Z_1 Z_2 Z_3 Z_4'],
    )
    return ec.CodeProfile(five_qubit_code).distance(errors='Z').value

**Why is $d_Z = 5$ even though the overall distance of the $[[5, 1, 3]]$ code is $d = 3$?**

Because the $[[5, 1, 3]]$ cyclic stabilizers mix $X$ and $Z$ operators, every weight-3 logical operator contains both $X$ and $Z$ Paulis. The minimum-weight logical operator consisting purely of $Z$ errors is $\bar{Z} = Z_0 Z_1 Z_2 Z_3 Z_4$ of weight `5`.

---

### 🤖 COPILOT INSTRUCTOR BRIEF — Module 1
- **Learning Objectives:**
  1. Construct a `qodec.Code` with `stabilizers`, `x`, and `z` lists in sparse `P_i` notation.
  2. Use `ec.CodeProfile(code)` to compute minimum-weight logical operators (`distance()`), noise-restricted distances (`distance(errors="Z")`), observable-specific distances (`distance(logical_observable=...)`), syndromes (`syndrome_of`), and logical effects (`logical_effect_of`).
- **Common Learner Pitfalls:**
  - Writing dense Pauli strings like `"XXXX"` instead of indexed sparse strings `"X_0 X_1 X_2 X_3"`.
  - Mis-ordering the `z` list so that $\bar{Z}_0$ commutes with $\bar{X}_0$ instead of anti-commuting. Point out that $\bar{X}_0 = X_0 X_1$ and $\bar{Z}_0 = Z_0 Z_2$ overlap ONLY on qubit `0`, so they anti-commute!
- **Socratic Prompt:** After running Cell 1 and Cell 2 below, ask: *"Why does `c4_profile.syndrome_of(ec.Pauli('X_1'))` return `{1}` while `syndrome_of(ec.Pauli('Z_2'))` returns `{0}`? Which stabilizer generator index (`0: X_0 X_1 X_2 X_3` vs `1: Z_0 Z_1 Z_2 Z_3`) detected each error?"*

#### 🤖 COPILOT TEACHING NOTE — Fire-and-Ice Extension (Module 1)
- Walk the learner through how `_build_fire_and_ice_20_2_6()` below uses the exact Lemma 1 + Corollary 2 construction from Reichardt, Aasen & Chao ([arXiv:2605.15344 ↗](https://arxiv.org/abs/2605.15344), Fig. 2):
  1. `inner_stabs`: 5 blocks of $[[4, 2, 2]]$ on qubits `0..3`, `4..7`, `8..11`, `12..15`, `16..19` (10 weight-4 stabilizers).
  2. `outer_stabs`: 4 cyclic $[[5, 1, 3]]$ generators lifted via Lemma 1 into 4 weight-8 $X$ stabilizers and 4 weight-8 $Z$ stabilizers (8 weight-8 stabilizers).
  3. Because the code is a self-dual CSS code, its minimum-weight logical operators are pure-$X$ or pure-$Z$ of weight $d_X = d_Z = 6$, which `fire_and_ice_profile.distance(errors="Z")` and `.distance(errors="X")` verify in $<0.1\text{ s}$!

<div style="display:flex;justify-content:space-between;align-items:stretch;gap:12px;margin-top:8px"><a href="../00-overview/overview.workbook.ipynb" title="Previous unit: Course Overview & QDK-EC Setup" style="display:inline-flex;align-items:center;gap:8px;padding:6px 12px;border:1px solid;border-radius:4px;text-decoration:none;background:var(--vscode-button-secondaryBackground, transparent);color:var(--vscode-button-secondaryForeground, inherit);border-color:var(--vscode-widget-border, #8884)"><span aria-hidden="true">&#8249;</span><span><span style="display:block;font-size:11px;opacity:.75">Previous unit</span><span style="display:block;font-weight:600">Course Overview & QDK-EC Setup</span></span></a><a href="../02-instruction-sets/instruction_sets.workbook.ipynb" title="Next unit: Build a Baseline Qodec & Instruction Set" style="display:inline-flex;align-items:center;gap:8px;padding:6px 12px;border:1px solid;border-radius:4px;text-decoration:none;background:var(--vscode-button-background, #005fb8);color:var(--vscode-button-foreground, #ffffff);border-color:var(--vscode-button-background, #005fb8)"><span style="text-align:right"><span style="display:block;font-size:11px;opacity:.75">Next unit</span><span style="display:block;font-weight:600">Build a Baseline Qodec & Instruction Set</span></span><span aria-hidden="true">&#8250;</span></a></div>